In [ ]:
# ==========================================
# PROJECT: YOLOv8 Traffic Light Detection
# AUTHOR: Ben H. Can Umutlu
# UNIVERSITY: Sivas Cumhuriyet University
# DESCRIPTION: Real-time traffic light detection 
#              trained on the BDD100K dataset.
# ==========================================

# 1. ENVIRONMENT SETUP
# Installing compatible libraries for the YOLOv8 and Kaggle integration
!pip uninstall -y kaggle kagglesdk kagglehub
!pip install -q kaggle kagglesdk kagglehub ultralytics

import os
import json
import glob
import random
import shutil
import yaml
from tqdm import tqdm
from PIL import Image as PILImage
from IPython.display import display
from ultralytics import YOLO

# 2. KAGGLE CREDENTIALS (SECURITY FIRST)
# IMPORTANT: Replace the placeholders below with your own Kaggle credentials.
# To keep your account safe, do not share your real keys on public repositories.
os.environ['KAGGLE_USERNAME'] = "YOUR_KAGGLE_USERNAME"
os.environ['KAGGLE_KEY'] = "YOUR_KAGGLE_API_KEY"

print("Environment ready! Connecting to Kaggle and downloading the BDD100K dataset...")

# 3. DATASET DOWNLOAD & EXTRACTION
# Downloading the BDD100K dataset (approx. 7.6 GB)
!kaggle datasets download -d solesensei/solesensei_bdd100k
!unzip -q solesensei_bdd100k.zip -d bdd100k_dataset

print("Dataset extraction complete.")

# 4. DATA PREPROCESSING (YOLO FORMAT CONVERSION)
# Converting BDD100K JSON annotations to YOLO .txt format (Normalized)
IMG_W, IMG_H = 1280.0, 720.0
color_map = {"red": 0, "green": 1, "yellow": 2}

output_dir = 'yolo_dataset/labels/train'
os.makedirs(output_dir, exist_ok=True)

# Finding the JSON label file automatically
json_files = glob.glob('bdd100k_dataset/**/*train*.json', recursive=True)
if json_files:
    json_path = [f for f in json_files if 'labels' in f or 'det' in f][0]
    
    with open(json_path, 'r') as f:
        data = json.load(f)

    for item in tqdm(data, desc="Converting to YOLO format"):
        img_name = item['name']
        labels = item.get('labels', [])
        yolo_lines = []

        for label in labels:
            if label['category'] == 'traffic light':
                color = label.get('attributes', {}).get('trafficLightColor', 'none')
                if color in color_map:
                    class_id = color_map[color]
                    box = label.get('box2d')
                    if box:
                        # Normalizing coordinates
                        cx = ((box['x1'] + box['x2']) / 2.0) / IMG_W
                        cy = ((box['y1'] + box['y2']) / 2.0) / IMG_H
                        w = (box['x2'] - box['x1']) / IMG_W
                        h = (box['y2'] - box['y1']) / IMG_H
                        yolo_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

        if yolo_lines:
            txt_name = os.path.splitext(img_name)[0] + ".txt"
            with open(os.path.join(output_dir, txt_name), 'w') as f:
                f.write("\n".join(yolo_lines))

# 5. MODEL TRAINING
# Initializing YOLOv8 Nano model for efficient training
model = YOLO('yolov8n.pt') 

# Creating data.yaml for YOLOv8
yaml_path = 'yolo_dataset/data.yaml'
yaml_data = {
    'path': '/content/yolo_dataset',
    'train': 'images/train',
    'val': 'images/train',
    'nc': 3,
    'names': ['Red', 'Green', 'Yellow']
}
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_data, f)

print("Starting training session (50 Epochs)...")
# Note: Results will be saved in 'Traffic_Lights/train_v1'
results = model.train(
    data=yaml_path,
    epochs=50,
    patience=10,
    imgsz=640,
    batch=16,
    device=0,
    project='Traffic_Lights',
    name='train_v1'
)

# 6. INFERENCE (PREDICTION)
# NOTE: To run this section after a fresh start, please use the weights 
# generated in the 'Traffic_Lights/train_v1/weights/best.pt' directory.
try:
    # Attempting to load the best weights from the most recent training
    model_path = 'Traffic_Lights/train_v1/weights/best.pt'
    model = YOLO(model_path)
    
    all_images = glob.glob('yolo_dataset/images/train/*.jpg')
    test_images = random.sample(all_images, 3)
    
    results = model.predict(source=test_images, conf=0.25)
    
    for r in results:
        im_array = r.plot()
        im_rgb = PILImage.fromarray(im_array[..., ::-1])
        display(im_rgb)
except Exception as e:
    print(f"Weight file not found. Please complete the training section first. Error: {e}")